In [1]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [2]:
from pathlib import Path
import json
import re
import sqlite3
from urllib.parse import unquote

import pandas as pd

In [3]:
import synth_extract.utils.sql_helpers as sql

In [4]:
sources = [
    {
        "source": "wiley",
        "db_path": "../data/scopus_wiley.db",
        "table": "papers_dup",
    },
    {
        "source": "springer_nature",
        "db_path": "../data/scopus_springer_nature.db",
        "table": "papers_dup",
    },
    {
        "source": "elsevier",
        "db_path": "../data/scopus_elsevier.db",
        "table": "papers_dup",
    },
    {
        "source": "s2orc",
        "db_path": "../data/s2orc.db",
        "table": "papers_dup",
    },
    {
        "source": "europepmc",
        "db_path": "../data/europepmc.db",
        "table": "papers_dup",
    },
    {
        "source": "arxiv",
        "db_path": "../data/arxiv.db",
        "table": "papers_dup",
    },
]

SOURCE_PRIORITY = {
    "springer_nature": 1,
    "wiley": 2,
    "elsevier": 3,
    "europepmc": 4,
    "s2orc": 5,
    "arxiv": 6,
}

CENTRAL_DB_PATH = "../data/central_papers.db"

# Number of records read from each source table at a time.
CHUNK_SIZE = 100_000

In [5]:
def quote_identifier(identifier):
    """
    Safely quote a SQLite table or column name.
    """
    return '"' + str(identifier).replace('"', '""') + '"'

#### Inspect DB Schemas

Should contain title, abstract and doi

In [6]:
def get_table_columns(db_path, table):
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            f"PRAGMA table_info({quote_identifier(table)})"
        ).fetchall()

    return [row[1] for row in rows]


schema_records = []
for source_config in sources:
    source_name = source_config["source"]
    db_path = source_config["db_path"]
    table = source_config["table"]

    path_exists = Path(db_path).exists()

    if path_exists:
        columns = get_table_columns(db_path, table)
    else:
        columns = []

    schema_records.append(
        {
            "source": source_name,
            "db_path": db_path,
            "table": table,
            "database_exists": path_exists,
            "columns": columns,
            "has_doi": "doi" in columns,
            "has_title": "title" in columns,
            "has_abstract": "abstract" in columns,
            "has_arxiv_id": "arxiv_id" in columns,
        }
    )

schema_df = pd.DataFrame(schema_records)

schema_df

,source,db_path,table,database_exists,columns,has_doi,has_title,has_abstract,has_arxiv_id
0,wiley,../data/scopus_wiley.db,papers_dup,True,"[eid, doi, title, journal, publisher, abstract]",True,True,True,False
1,springer_nature,../data/scopus_springer_nature.db,papers_dup,True,"[eid, doi, title, journal, publisher, abstract]",True,True,True,False
2,elsevier,../data/scopus_elsevier.db,papers_dup,True,"[eid, doi, title, journal, publisher, abstract]",True,True,True,False
3,s2orc,../data/s2orc.db,papers_dup,True,"[corpusid, doi, title, authors, abstract]",True,True,True,False
4,europepmc,../data/europepmc.db,papers_dup,True,"[pmcid, pmid, doi, title, journal, abstract]",True,True,True,False
5,arxiv,../data/arxiv.db,papers_dup,True,"[arxiv_id, version, arxiv_url, doi, title, abs...",True,True,True,True


In [7]:
required_common_columns = {"doi", "title", "abstract"}

schema_errors = []

for record in schema_records:
    source_name = record["source"]
    columns = set(record["columns"])

    if not record["database_exists"]:
        schema_errors.append(
            f"{source_name}: database does not exist: {record['db_path']}"
        )
        continue

    missing = required_common_columns - columns

    if missing:
        schema_errors.append(
            f"{source_name}: missing columns {sorted(missing)}"
        )

    if source_name == "arxiv" and "arxiv_id" not in columns:
        schema_errors.append(
            "arxiv: expected an 'arxiv_id' column"
        )

if schema_errors:
    raise ValueError(
        "Source schema validation failed:\n- "
        + "\n- ".join(schema_errors)
    )

print("All source schemas passed validation.")

All source schemas passed validation.


#### Collect source-level statistics

In [8]:
def scalar_query(conn, query):
    return conn.execute(query).fetchone()[0]

In [9]:
source_stats = []

for source_config in sources:
    source_name = source_config["source"]
    db_path = source_config["db_path"]
    table = quote_identifier(source_config["table"])

    with sqlite3.connect(db_path) as conn:
        total_rows = scalar_query(
            conn,
            f"SELECT COUNT(*) FROM {table}"
        )

        non_null_doi_rows = scalar_query(
            conn,
            f"""
            SELECT COUNT(*)
            FROM {table}
            WHERE doi IS NOT NULL
              AND TRIM(CAST(doi AS TEXT)) <> ''
            """
        )

        raw_unique_dois = scalar_query(
            conn,
            f"""
            SELECT COUNT(DISTINCT LOWER(TRIM(CAST(doi AS TEXT))))
            FROM {table}
            WHERE doi IS NOT NULL
              AND TRIM(CAST(doi AS TEXT)) <> ''
            """
        )

        raw_duplicate_doi_groups = scalar_query(
            conn,
            f"""
            SELECT COUNT(*)
            FROM (
                SELECT LOWER(TRIM(CAST(doi AS TEXT))) AS doi_value
                FROM {table}
                WHERE doi IS NOT NULL
                  AND TRIM(CAST(doi AS TEXT)) <> ''
                GROUP BY LOWER(TRIM(CAST(doi AS TEXT)))
                HAVING COUNT(*) > 1
            )
            """
        )

        null_doi_rows = total_rows - non_null_doi_rows

        stats = {
            "source": source_name,
            "total_rows": total_rows,
            "rows_with_doi": non_null_doi_rows,
            "rows_without_doi": null_doi_rows,
            "raw_unique_dois": raw_unique_dois,
            "raw_duplicate_doi_groups": raw_duplicate_doi_groups,
        }

        if source_name == "arxiv":
            arxiv_id_column = quote_identifier(
                source_config.get(
                    "arxiv_id_column",
                    "arxiv_id",
                )
            )

            stats["rows_with_arxiv_id"] = scalar_query(
                conn,
                f"""
                SELECT COUNT(*)
                FROM {table}
                WHERE {arxiv_id_column} IS NOT NULL
                  AND TRIM(CAST({arxiv_id_column} AS TEXT)) <> ''
                """
            )

            stats["raw_unique_arxiv_ids"] = scalar_query(
                conn,
                f"""
                SELECT COUNT(
                    DISTINCT LOWER(TRIM(CAST({arxiv_id_column} AS TEXT)))
                )
                FROM {table}
                WHERE {arxiv_id_column} IS NOT NULL
                  AND TRIM(CAST({arxiv_id_column} AS TEXT)) <> ''
                """
            )

            stats["arxiv_without_doi"] = scalar_query(
                conn,
                f"""
                SELECT COUNT(*)
                FROM {table}
                WHERE (
                    doi IS NULL
                    OR TRIM(CAST(doi AS TEXT)) = ''
                )
                AND {arxiv_id_column} IS NOT NULL
                AND TRIM(CAST({arxiv_id_column} AS TEXT)) <> ''
                """
            )

        source_stats.append(stats)

source_stats_df = pd.DataFrame(source_stats)

In [10]:
print(f"Total source rows: {source_stats_df['total_rows'].sum():,}")
print(
    f"Rows containing a DOI: "
    f"{source_stats_df['rows_with_doi'].sum():,}"
)
print(
    f"Rows without a DOI: "
    f"{source_stats_df['rows_without_doi'].sum():,}"
)

display(source_stats_df)

Total source rows: 1,347,478
Rows containing a DOI: 1,341,749
Rows without a DOI: 5,729


,source,total_rows,rows_with_doi,rows_without_doi,raw_unique_dois,raw_duplicate_doi_groups,rows_with_arxiv_id,raw_unique_arxiv_ids,arxiv_without_doi
0,wiley,189418,189418,0,189418,0,NaN,NaN,NaN
1,springer_nature,14913,14913,0,14913,0,NaN,NaN,NaN
2,elsevier,499800,499800,0,499800,0,NaN,NaN,NaN
3,s2orc,348389,348389,0,348389,0,NaN,NaN,NaN
4,europepmc,280477,280477,0,280477,0,NaN,NaN,NaN
5,arxiv,14481,8752,5729,8752,0,14481.0,14481.0,5729.0


This matches what we already have. There are no duplicates within a source. 

#### Create the central database and staging table

In [11]:
central_db_path = Path(CENTRAL_DB_PATH)
central_db_path.parent.mkdir(parents=True, exist_ok=True)

if central_db_path.exists():
    central_db_path.unlink()

print(f"Creating central database: {central_db_path}")

Creating central database: ../data/central_papers.db


In [12]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA synchronous = NORMAL")
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA foreign_keys = ON")

    conn.execute("""
        CREATE TABLE staging_records (
            staging_id INTEGER PRIMARY KEY,

            source TEXT NOT NULL,
            source_priority INTEGER NOT NULL,

            doi TEXT,
            arxiv_id TEXT,

            identifier_type TEXT NOT NULL,
            identifier_value TEXT NOT NULL,
            identifier_key TEXT NOT NULL,

            title TEXT,
            abstract TEXT
        )
    """)

    conn.execute("""
        CREATE INDEX idx_staging_identifier_key
        ON staging_records(identifier_key)
    """)

    conn.execute("""
        CREATE INDEX idx_staging_source
        ON staging_records(source)
    """)

    conn.execute("""
        CREATE INDEX idx_staging_doi
        ON staging_records(doi)
    """)

    conn.execute("""
        CREATE INDEX idx_staging_arxiv_id
        ON staging_records(arxiv_id)
    """)

    conn.commit()

#### Load normalized source records into staging

The staging table is a temporary workspace used to consolidate records from all source databases before constructing the final central database.

Each source record is assigned a single canonical identifier:

- If a DOI is present, it is used as the paper identifier.
- If the DOI is missing (possible only for arXiv), the arXiv ID is used.
- Records with neither a DOI nor an arXiv ID are discarded. This should never occur in the cleaned source databases, so a non-zero `dropped_missing_identifier` count indicates a data quality issue.

Each staged record contains:

- Source name
- Source priority
- DOI
- arXiv ID (if available)
- Identifier type (`doi` or `arxiv_id`)
- Identifier value
- Combined identifier key (`doi:<doi>` or `arxiv_id:<arxiv_id>`)
- Title
- Abstract

The `identifier_key` uniquely identifies a paper across all sources and is used in the next stage to group duplicate records and construct the final central database.

In [13]:
ingestion_stats = []

with sqlite3.connect(CENTRAL_DB_PATH) as central_conn:
    central_conn.execute("PRAGMA journal_mode = WAL")
    central_conn.execute("PRAGMA synchronous = NORMAL")

    for source_config in sources:
        source_name = source_config["source"]
        source_priority = SOURCE_PRIORITY[source_name]
        db_path = source_config["db_path"]
        table = quote_identifier(source_config["table"])

        if source_name == "arxiv":
            arxiv_id_column = quote_identifier(
                "arxiv_id"
            )

            query = f"""
                SELECT
                    doi,
                    {arxiv_id_column} AS arxiv_id,
                    title,
                    abstract
                FROM {table}
            """
        else:
            query = f"""
                SELECT
                    doi,
                    NULL AS arxiv_id,
                    title,
                    abstract
                FROM {table}
            """

        input_rows = 0
        staged_rows = 0
        dropped_missing_identifier = 0
        # invalid_doi_rows = 0

        print(f"Loading {source_name}...")

        with sqlite3.connect(db_path) as source_conn:
            chunks = pd.read_sql_query(
                query,
                source_conn,
                chunksize=CHUNK_SIZE,
            )

            for chunk in chunks:
                input_rows += len(chunk)

                # raw_doi_present = (
                #     chunk["doi"].notna()
                #     & chunk["doi"].astype(str).str.strip().ne("")
                # )

                # invalid_doi_rows += int(
                #     (
                #         raw_doi_present
                #         & chunk["doi"].isna()
                #     ).sum()
                # )

                # DOI is the preferred identity whenever it is available.
                chunk["identifier_type"] = "doi"
                chunk["identifier_value"] = chunk["doi"]

                # DOI-less arXiv records use the arXiv ID.
                use_arxiv_identity = (
                    chunk["doi"].isna()
                    & chunk["arxiv_id"].notna()
                )

                chunk.loc[
                    use_arxiv_identity,
                    "identifier_type",
                ] = "arxiv_id"

                chunk.loc[
                    use_arxiv_identity,
                    "identifier_value",
                ] = chunk.loc[
                    use_arxiv_identity,
                    "arxiv_id",
                ]

                missing_identifier = chunk["identifier_value"].isna()
                dropped_missing_identifier += int(
                    missing_identifier.sum()
                )

                chunk = chunk.loc[~missing_identifier].copy()

                chunk["identifier_key"] = (
                    chunk["identifier_type"]
                    + ":"
                    + chunk["identifier_value"]
                )

                chunk["source"] = source_name
                chunk["source_priority"] = source_priority

                output_columns = [
                    "source",
                    "source_priority",
                    "doi",
                    "arxiv_id",
                    "identifier_type",
                    "identifier_value",
                    "identifier_key",
                    "title",
                    "abstract",
                ]

                chunk[output_columns].to_sql(
                    "staging_records",
                    central_conn,
                    if_exists="append",
                    index=False,
                    method="multi",
                    chunksize=5_000,
                )

                staged_rows += len(chunk)

        central_conn.commit()

        ingestion_stats.append(
            {
                "source": source_name,
                "input_rows": input_rows,
                "staged_rows": staged_rows,
                "dropped_missing_identifier": (
                    dropped_missing_identifier
                ),
                # "invalid_doi_rows": invalid_doi_rows,
            }
        )

        print(
            f"  Input: {input_rows:,} | "
            f"Staged: {staged_rows:,} | "
            f"Dropped: {dropped_missing_identifier:,} | "
            # f"Invalid DOI values: {invalid_doi_rows:,}"
        )

Loading wiley...
  Input: 189,418 | Staged: 189,418 | Dropped: 0 | 
Loading springer_nature...
  Input: 14,913 | Staged: 14,913 | Dropped: 0 | 
Loading elsevier...
  Input: 499,800 | Staged: 499,800 | Dropped: 0 | 
Loading s2orc...
  Input: 348,389 | Staged: 348,389 | Dropped: 0 | 
Loading europepmc...
  Input: 280,477 | Staged: 280,477 | Dropped: 0 | 
Loading arxiv...
  Input: 14,481 | Staged: 14,481 | Dropped: 0 | 


In [14]:
ingestion_stats_df = pd.DataFrame(ingestion_stats)

ingestion_stats_df

,source,input_rows,staged_rows,dropped_missing_identifier
0,wiley,189418,189418,0
1,springer_nature,14913,14913,0
2,elsevier,499800,499800,0
3,s2orc,348389,348389,0
4,europepmc,280477,280477,0
5,arxiv,14481,14481,0


#### Look at the Stage Table

In [15]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    prebuild_stats = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS staged_source_records,

            SUM(
                CASE
                    WHEN identifier_type = 'doi' THEN 1
                    ELSE 0
                END
            ) AS source_records_using_doi,

            SUM(
                CASE
                    WHEN identifier_type = 'arxiv_id' THEN 1
                    ELSE 0
                END
            ) AS source_records_using_arxiv_id,

            COUNT(
                DISTINCT CASE
                    WHEN identifier_type = 'doi'
                    THEN identifier_value
                END
            ) AS unique_dois,

            COUNT(
                DISTINCT CASE
                    WHEN identifier_type = 'arxiv_id'
                    THEN identifier_value
                END
            ) AS unique_doi_less_arxiv_ids,

            COUNT(DISTINCT identifier_key)
                AS expected_unique_papers

        FROM staging_records
        """,
        conn,
    )

prebuild_stats.T

,0
staged_source_records,1347478
source_records_using_doi,1341749
source_records_using_arxiv_id,5729
unique_dois,1096180
unique_doi_less_arxiv_ids,5729
expected_unique_papers,1101909


In [16]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    normalized_source_stats = pd.read_sql_query(
        """
        SELECT
            source,
            COUNT(*) AS staged_rows,
            COUNT(DISTINCT identifier_key) AS unique_papers,
            COUNT(DISTINCT doi) AS unique_dois,
            COUNT(DISTINCT arxiv_id) AS unique_arxiv_ids
        FROM staging_records
        GROUP BY source
        ORDER BY source_priority
        """,
        conn,
    )

normalized_source_stats

,source,staged_rows,unique_papers,unique_dois,unique_arxiv_ids
0,springer_nature,14913,14913,14913,0
1,wiley,189418,189418,189418,0
2,elsevier,499800,499800,499800,0
3,europepmc,280477,280477,280477,0
4,s2orc,348389,348389,348389,0
5,arxiv,14481,14481,8752,14481


In [17]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    doi_overlap_stats = pd.read_sql_query(
        """
        WITH doi_groups AS (
            SELECT
                doi,
                COUNT(*) AS record_count,
                COUNT(DISTINCT source) AS source_count
            FROM staging_records
            WHERE doi IS NOT NULL
            GROUP BY doi
        )
        SELECT
            COUNT(*) AS unique_dois,
            SUM(
                CASE WHEN source_count = 1 THEN 1 ELSE 0 END
            ) AS dois_in_one_source,
            SUM(
                CASE WHEN source_count >= 2 THEN 1 ELSE 0 END
            ) AS dois_in_multiple_sources,
            MAX(source_count) AS maximum_sources_per_doi
        FROM doi_groups
        """,
        conn,
    )

doi_overlap_stats.T

,0
unique_dois,1096180
dois_in_one_source,868395
dois_in_multiple_sources,227785
maximum_sources_per_doi,4


In [18]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    source_count_distribution = pd.read_sql_query(
        """
        WITH paper_groups AS (
            SELECT
                identifier_key,
                COUNT(DISTINCT source) AS source_count
            FROM staging_records
            GROUP BY identifier_key
        )
        SELECT
            source_count,
            COUNT(*) AS paper_count
        FROM paper_groups
        GROUP BY source_count
        ORDER BY source_count
        """,
        conn,
    )

source_count_distribution

,source_count,paper_count
0,1,874124
1,2,210040
2,3,17706
3,4,39


In [19]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    source_combination_stats = pd.read_sql_query(
        """
        WITH distinct_sources AS (
            SELECT DISTINCT
                identifier_key,
                source,
                source_priority
            FROM staging_records
        ),
        ordered_sources AS (
            SELECT
                identifier_key,
                source,
                source_priority
            FROM distinct_sources
            ORDER BY
                identifier_key,
                source_priority
        ),
        combinations AS (
            SELECT
                identifier_key,
                GROUP_CONCAT(source, ' -> ') AS source_combination
            FROM ordered_sources
            GROUP BY identifier_key
        )
        SELECT
            source_combination,
            COUNT(*) AS paper_count
        FROM combinations
        GROUP BY source_combination
        ORDER BY paper_count DESC
        """,
        conn,
    )

source_combination_stats

,source_combination,paper_count
0,elsevier,481378
1,europepmc -> s2orc,183099
2,wiley,172147
3,s2orc,129459
4,europepmc,71734
5,springer_nature,13213
6,wiley -> europepmc -> s2orc,7823
7,elsevier -> europepmc -> s2orc,7702
8,s2orc -> arxiv,6833
9,arxiv,6193


#### Create Final Table

In [20]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS papers")

    conn.execute("""
        CREATE TABLE papers (
            paper_id INTEGER PRIMARY KEY,
            paper_uid TEXT NOT NULL UNIQUE,

            identifier_type TEXT NOT NULL
                CHECK (
                    identifier_type IN ('doi', 'arxiv_id')
                ),

            identifier_value TEXT NOT NULL UNIQUE,

            doi TEXT,
            arxiv_id TEXT,

            title TEXT,
            abstract TEXT,

            sources TEXT NOT NULL,
            source_count INTEGER NOT NULL,

            source_order TEXT NOT NULL,

            canonical_source TEXT NOT NULL,
            canonical_source_position INTEGER NOT NULL DEFAULT 1,

            download_status TEXT NOT NULL DEFAULT 'pending'
                CHECK (
                    download_status IN (
                        'pending',
                        'success',
                        'failed',
                        'exhausted'
                    )
                ),

            attempt_count INTEGER NOT NULL DEFAULT 0,

            attempted_sources TEXT NOT NULL DEFAULT '[]',
            failure_history TEXT NOT NULL DEFAULT '[]',

            last_attempted_source TEXT,
            last_attempted_at TEXT,
            last_error TEXT,

            downloaded_from TEXT,
            fulltext_path TEXT,
            fulltext_format TEXT,
            downloaded_at TEXT,

            review_note TEXT,

            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
            updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

            CHECK (
                (identifier_type = 'doi' AND doi IS NOT NULL)
                OR
                (
                    identifier_type = 'arxiv_id'
                    AND arxiv_id IS NOT NULL
                )
            )
        )
    """)

    conn.execute("""
        CREATE UNIQUE INDEX idx_papers_uid
        ON papers(paper_uid)
    """)

    conn.execute("""
        CREATE UNIQUE INDEX idx_papers_identifier
        ON papers(identifier_type, identifier_value)
    """)

    conn.execute("""
        CREATE UNIQUE INDEX idx_papers_doi
        ON papers(doi)
        WHERE doi IS NOT NULL
    """)

    conn.execute("""
        CREATE UNIQUE INDEX idx_papers_arxiv_only
        ON papers(arxiv_id)
        WHERE identifier_type = 'arxiv_id'
          AND arxiv_id IS NOT NULL
    """)

    conn.execute("""
        CREATE INDEX idx_papers_download_status
        ON papers(download_status)
    """)

    conn.execute("""
        CREATE INDEX idx_papers_canonical_source
        ON papers(canonical_source)
    """)

    conn.commit()

#### Populate the Central Papers Table

This step constructs the final `papers` table from the staged source records by creating a single canonical record for each unique paper.

The process:

- Groups staged records by `identifier_key`.
- Creates one row for each unique DOI or DOI-less arXiv ID.
- Selects the highest-priority available source as the `canonical_source`.
- Records all available sources in deterministic priority order.
- Selects the highest-priority non-empty title and abstract.
- Assigns a unique `paper_id` and human-readable `paper_uid`.
- Initializes the manual download workflow by setting:
  - `download_status = 'pending'`
  - `attempt_count = 0`
  - `attempted_sources = []`
  - `failure_history = []`

The resulting `papers` table serves as the central registry for all unique papers and is used to drive the subsequent full-text download workflow.

Suppose the following records exist in the staging table:

| Source | DOI | arXiv ID | Title |
|--------|-----|----------|-------|
| Springer Nature | 10.1000/abc123 | - | Paper Title |
| Europe PMC | 10.1000/abc123 | - | Paper Title |
| S2ORC | 10.1000/abc123 | - | Paper Title |
| arXiv | 10.1000/abc123 | 2401.12345 | Paper Title |

All four records correspond to the same paper because they share the same DOI. They are merged into a single row in the `papers` table:

| paper_uid | identifier_type | identifier_value | canonical_source | source_order | source_count | download_status |
|-----------|-----------------|------------------|------------------|--------------|--------------|-----------------|
| ID000000001 | doi | 10.1000/abc123 | springer_nature | ["springer_nature", "europepmc", "s2orc", "arxiv"] | 4 | pending |

The downloader will first attempt to retrieve the full text from **Springer Nature**. If that attempt fails, the `canonical_source` can be manually updated to the next source in `source_order` (Europe PMC), and the download can be retried. This process continues until a successful download is obtained or all available sources have been exhausted.

---

For a DOI-less arXiv paper:

| Source | DOI | arXiv ID | Title |
|--------|-----|----------|-------|
| arXiv | - | 2501.01234 | Another Paper |

Since no DOI is available, the arXiv ID becomes the canonical identifier:

| paper_uid | identifier_type | identifier_value | canonical_source | source_order | source_count | download_status |
|-----------|-----------------|------------------|------------------|--------------|--------------|-----------------|
| ID000000002 | arxiv_id | 2501.01234 | arxiv | ["arxiv"] | 1 | pending |

In [21]:
populate_sql = """
WITH

distinct_source_records AS (
    SELECT
        identifier_key,
        identifier_type,
        identifier_value,
        doi,
        arxiv_id,
        source,
        source_priority,
        title,
        abstract
    FROM (
        SELECT
            sr.*,
            ROW_NUMBER() OVER (
                PARTITION BY identifier_key, source
                ORDER BY staging_id
            ) AS source_row_number
        FROM staging_records AS sr
    )
    WHERE source_row_number = 1
),

paper_base AS (
    SELECT
        identifier_key,
        MIN(identifier_type) AS identifier_type,
        MIN(identifier_value) AS identifier_value,
        MAX(doi) AS doi,
        MAX(arxiv_id) AS arxiv_id,
        COUNT(DISTINCT source) AS source_count
    FROM distinct_source_records
    GROUP BY identifier_key
),

canonical_source_rows AS (
    SELECT
        identifier_key,
        source AS canonical_source
    FROM (
        SELECT
            identifier_key,
            source,
            ROW_NUMBER() OVER (
                PARTITION BY identifier_key
                ORDER BY source_priority, source
            ) AS source_rank
        FROM distinct_source_records
    )
    WHERE source_rank = 1
),

best_title_rows AS (
    SELECT
        identifier_key,
        title
    FROM (
        SELECT
            identifier_key,
            title,
            ROW_NUMBER() OVER (
                PARTITION BY identifier_key
                ORDER BY
                    CASE
                        WHEN title IS NULL OR TRIM(title) = ''
                        THEN 1
                        ELSE 0
                    END,
                    source_priority,
                    source
            ) AS title_rank
        FROM distinct_source_records
    )
    WHERE title_rank = 1
),

best_abstract_rows AS (
    SELECT
        identifier_key,
        abstract
    FROM (
        SELECT
            identifier_key,
            abstract,
            ROW_NUMBER() OVER (
                PARTITION BY identifier_key
                ORDER BY
                    CASE
                        WHEN abstract IS NULL OR TRIM(abstract) = ''
                        THEN 1
                        ELSE 0
                    END,
                    source_priority,
                    source
            ) AS abstract_rank
        FROM distinct_source_records
    )
    WHERE abstract_rank = 1
),

ordered_distinct_sources AS (
    SELECT
        identifier_key,
        source,
        source_priority
    FROM (
        SELECT DISTINCT
            identifier_key,
            source,
            source_priority
        FROM distinct_source_records
    )
    ORDER BY
        identifier_key,
        source_priority,
        source
),

source_lists AS (
    SELECT
        identifier_key,

        '["'
        || GROUP_CONCAT(source, '","')
        || '"]' AS ordered_source_json

    FROM ordered_distinct_sources
    GROUP BY identifier_key
),

assembled AS (
    SELECT
        pb.identifier_key,
        pb.identifier_type,
        pb.identifier_value,
        pb.doi,
        pb.arxiv_id,
        bt.title,
        ba.abstract,
        sl.ordered_source_json,
        pb.source_count,
        csr.canonical_source
    FROM paper_base AS pb

    JOIN canonical_source_rows AS csr
      ON csr.identifier_key = pb.identifier_key

    JOIN source_lists AS sl
      ON sl.identifier_key = pb.identifier_key

    LEFT JOIN best_title_rows AS bt
      ON bt.identifier_key = pb.identifier_key

    LEFT JOIN best_abstract_rows AS ba
      ON ba.identifier_key = pb.identifier_key
),

numbered AS (
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY
                CASE identifier_type
                    WHEN 'doi' THEN 1
                    ELSE 2
                END,
                identifier_value
        ) AS paper_id,

        *
    FROM assembled
)

INSERT INTO papers (
    paper_id,
    paper_uid,
    identifier_type,
    identifier_value,
    doi,
    arxiv_id,
    title,
    abstract,
    sources,
    source_count,
    source_order,
    canonical_source,
    canonical_source_position,
    download_status,
    attempt_count,
    attempted_sources,
    failure_history
)

SELECT
    paper_id,
    printf('ID%09d', paper_id) AS paper_uid,
    identifier_type,
    identifier_value,
    doi,
    arxiv_id,
    title,
    abstract,

    ordered_source_json AS sources,
    source_count,
    ordered_source_json AS source_order,

    canonical_source,
    1 AS canonical_source_position,

    'pending' AS download_status,
    0 AS attempt_count,
    '[]' AS attempted_sources,
    '[]' AS failure_history

FROM numbered
ORDER BY paper_id
"""

In [22]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA synchronous = NORMAL")
    conn.execute("PRAGMA temp_store = MEMORY")

    conn.execute(populate_sql)
    conn.commit()

print("Central papers table populated.")

Central papers table populated.


#### Inspect papers table

In [23]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    sample_papers = pd.read_sql_query(
        """
        SELECT
            paper_id,
            paper_uid,
            identifier_type,
            identifier_value,
            doi,
            arxiv_id,
            canonical_source,
            source_count,
            source_order,
            download_status,
            title
        FROM papers
        ORDER BY paper_id
        LIMIT 20
        """,
        conn,
    )

sample_papers

,paper_id,paper_uid,identifier_type,identifier_value,doi,arxiv_id,canonical_source,source_count,source_order,download_status,title
0,1,ID000000001,doi,10.1001/2013.jamaneurol.537,10.1001/2013.jamaneurol.537,None,europepmc,1,"[""europepmc""]",pending,C9orf72 hexanucleotide repeat expansions in cl...
1,2,ID000000002,doi,10.1001/archderm.139.10.1325,10.1001/archderm.139.10.1325,None,s2orc,1,"[""s2orc""]",pending,Mechanisms underlying imiquimod-induced regres...
2,3,ID000000003,doi,10.1001/archderm.139.4.498,10.1001/archderm.139.4.498,None,s2orc,1,"[""s2orc""]",pending,Epidermolysis bullosa simplex in Israel: clini...
3,4,ID000000004,doi,10.1001/archderm.144.7.851,10.1001/archderm.144.7.851,None,europepmc,1,"[""europepmc""]",pending,Effect of increased pigmentation on the antifi...
4,5,ID000000005,doi,10.1001/archderm.144.7.902,10.1001/archderm.144.7.902,None,europepmc,1,"[""europepmc""]",pending,Acute skin eruptions that are positive for her...
5,6,ID000000006,doi,10.1001/archdermatol.2010.86,10.1001/archdermatol.2010.86,None,europepmc,1,"[""europepmc""]",pending,Pyoderma gangrenosum-like ulcer in a patient w...
6,7,ID000000007,doi,10.1001/archdermatol.2011.1413,10.1001/archdermatol.2011.1413,None,europepmc,2,"[""europepmc"",""s2orc""]",pending,Viral-associated trichodysplasia: characteriza...
7,8,ID000000008,doi,10.1001/archdermatol.2011.187,10.1001/archdermatol.2011.187,None,europepmc,1,"[""europepmc""]",pending,Clonal T-cell receptor β-chain gene rearrangem...
8,9,ID000000009,doi,10.1001/archgenpsychiatry.2010.114,10.1001/archgenpsychiatry.2010.114,None,europepmc,1,"[""europepmc""]",pending,Altered expression of regulators of the cortic...
9,10,ID000000010,doi,10.1001/archgenpsychiatry.2010.175,10.1001/archgenpsychiatry.2010.175,None,europepmc,1,"[""europepmc""]",pending,Hippocampal interneurons in bipolar disorder.


In [24]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    central_stats = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_unique_papers,

            SUM(
                CASE WHEN identifier_type = 'doi'
                THEN 1 ELSE 0 END
            ) AS doi_identified_papers,

            SUM(
                CASE WHEN identifier_type = 'arxiv_id'
                THEN 1 ELSE 0 END
            ) AS doi_less_arxiv_papers,

            COUNT(doi) AS papers_with_doi,

            SUM(
                CASE WHEN arxiv_id IS NOT NULL
                THEN 1 ELSE 0 END
            ) AS papers_with_arxiv_id,

            SUM(
                CASE WHEN source_count = 1
                THEN 1 ELSE 0 END
            ) AS papers_in_one_source,

            SUM(
                CASE WHEN source_count >= 2
                THEN 1 ELSE 0 END
            ) AS papers_in_multiple_sources,

            MAX(source_count) AS maximum_source_count

        FROM papers
        """,
        conn,
    )

central_stats.T

,0
total_unique_papers,1101909
doi_identified_papers,1096180
doi_less_arxiv_papers,5729
papers_with_doi,1096180
papers_with_arxiv_id,14481
papers_in_one_source,874124
papers_in_multiple_sources,227785
maximum_source_count,4


In [25]:
stats = central_stats.iloc[0]

print(f"Total unique papers: {stats['total_unique_papers']:,}")
print(
    f"DOI-identified papers: "
    f"{stats['doi_identified_papers']:,}"
)
print(
    f"DOI-less arXiv papers: "
    f"{stats['doi_less_arxiv_papers']:,}"
)
print(
    f"Papers found in one source: "
    f"{stats['papers_in_one_source']:,}"
)
print(
    f"Papers found in multiple sources: "
    f"{stats['papers_in_multiple_sources']:,}"
)
print(
    f"Maximum source count for one paper: "
    f"{stats['maximum_source_count']:,}"
)

Total unique papers: 1,101,909
DOI-identified papers: 1,096,180
DOI-less arXiv papers: 5,729
Papers found in one source: 874,124
Papers found in multiple sources: 227,785
Maximum source count for one paper: 4


In [26]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    canonical_source_stats = pd.read_sql_query(
        """
        SELECT
            canonical_source,
            COUNT(*) AS paper_count,
            ROUND(
                100.0 * COUNT(*) /
                (SELECT COUNT(*) FROM papers),
                2
            ) AS percentage
        FROM papers
        GROUP BY canonical_source
        ORDER BY
            CASE canonical_source
                WHEN 'springer_nature' THEN 1
                WHEN 'wiley' THEN 2
                WHEN 'elsevier' THEN 3
                WHEN 'europepmc' THEN 4
                WHEN 's2orc' THEN 5
                WHEN 'arxiv' THEN 6
                ELSE 99
            END
        """,
        conn,
    )

canonical_source_stats

,canonical_source,paper_count,percentage
0,springer_nature,14913,1.35
1,wiley,189418,17.19
2,elsevier,499644,45.34
3,europepmc,255449,23.18
4,s2orc,136292,12.37
5,arxiv,6193,0.56


In [27]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    central_source_count_stats = pd.read_sql_query(
        """
        SELECT
            source_count,
            COUNT(*) AS paper_count,
            ROUND(
                100.0 * COUNT(*) /
                (SELECT COUNT(*) FROM papers),
                2
            ) AS percentage
        FROM papers
        GROUP BY source_count
        ORDER BY source_count
        """,
        conn,
    )

central_source_count_stats

,source_count,paper_count,percentage
0,1,874124,79.33
1,2,210040,19.06
2,3,17706,1.61
3,4,39,0.00


In [30]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    central_combination_stats = pd.read_sql_query(
        """
        SELECT
            source_order,
            COUNT(*) AS paper_count
        FROM papers
        GROUP BY source_order
        ORDER BY paper_count DESC
        """,
        conn,
    )

central_combination_stats

,source_order,paper_count
0,"[""elsevier""]",481378
1,"[""europepmc"",""s2orc""]",183099
2,"[""wiley""]",172147
3,"[""s2orc""]",129459
4,"[""europepmc""]",71734
5,"[""springer_nature""]",13213
6,"[""wiley"",""europepmc"",""s2orc""]",7823
7,"[""elsevier"",""europepmc"",""s2orc""]",7702
8,"[""s2orc"",""arxiv""]",6833
9,"[""arxiv""]",6193


In [31]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    download_state_stats = pd.read_sql_query(
        """
        SELECT
            download_status,
            COUNT(*) AS paper_count,
            ROUND(
                100.0 * COUNT(*) /
                (SELECT COUNT(*) FROM papers),
                2
            ) AS percentage
        FROM papers
        GROUP BY download_status
        ORDER BY download_status
        """,
        conn,
    )

download_state_stats

,download_status,paper_count,percentage
0,pending,1101909,100.0


In [32]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    source_status_stats = pd.read_sql_query(
        """
        SELECT
            canonical_source,
            download_status,
            COUNT(*) AS paper_count
        FROM papers
        GROUP BY
            canonical_source,
            download_status
        ORDER BY
            CASE canonical_source
                WHEN 'springer_nature' THEN 1
                WHEN 'wiley' THEN 2
                WHEN 'elsevier' THEN 3
                WHEN 'europepmc' THEN 4
                WHEN 's2orc' THEN 5
                WHEN 'arxiv' THEN 6
                ELSE 99
            END,
            download_status
        """,
        conn,
    )

source_status_stats

,canonical_source,download_status,paper_count
0,springer_nature,pending,14913
1,wiley,pending,189418
2,elsevier,pending,499644
3,europepmc,pending,255449
4,s2orc,pending,136292
5,arxiv,pending,6193


In [33]:
validation_queries = {
    "duplicate_paper_uid": """
        SELECT COUNT(*)
        FROM (
            SELECT paper_uid
            FROM papers
            GROUP BY paper_uid
            HAVING COUNT(*) > 1
        )
    """,

    "duplicate_identifier": """
        SELECT COUNT(*)
        FROM (
            SELECT identifier_type, identifier_value
            FROM papers
            GROUP BY identifier_type, identifier_value
            HAVING COUNT(*) > 1
        )
    """,

    "duplicate_non_null_doi": """
        SELECT COUNT(*)
        FROM (
            SELECT doi
            FROM papers
            WHERE doi IS NOT NULL
            GROUP BY doi
            HAVING COUNT(*) > 1
        )
    """,

    "missing_required_identifier": """
        SELECT COUNT(*)
        FROM papers
        WHERE identifier_value IS NULL
           OR identifier_type IS NULL
    """,

    "invalid_doi_identity": """
        SELECT COUNT(*)
        FROM papers
        WHERE identifier_type = 'doi'
          AND doi IS NULL
    """,

    "invalid_arxiv_identity": """
        SELECT COUNT(*)
        FROM papers
        WHERE identifier_type = 'arxiv_id'
          AND arxiv_id IS NULL
    """,

    "invalid_canonical_source": """
        SELECT COUNT(*)
        FROM papers
        WHERE canonical_source IS NULL
           OR canonical_source = ''
    """,

    "invalid_source_count": """
        SELECT COUNT(*)
        FROM papers
        WHERE source_count < 1
    """,

    "unexpected_initial_status": """
        SELECT COUNT(*)
        FROM papers
        WHERE download_status <> 'pending'
    """,
}

In [34]:
validation_results = []

with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    for check_name, query in validation_queries.items():
        issue_count = conn.execute(query).fetchone()[0]

        validation_results.append(
            {
                "check": check_name,
                "issue_count": issue_count,
                "passed": issue_count == 0,
            }
        )

validation_df = pd.DataFrame(validation_results)

validation_df

,check,issue_count,passed
0,duplicate_paper_uid,0,True
1,duplicate_identifier,0,True
2,duplicate_non_null_doi,0,True
3,missing_required_identifier,0,True
4,invalid_doi_identity,0,True
5,invalid_arxiv_identity,0,True
6,invalid_canonical_source,0,True
7,invalid_source_count,0,True
8,unexpected_initial_status,0,True


In [35]:
with sqlite3.connect(CENTRAL_DB_PATH) as conn:
    source_order_issues = pd.read_sql_query(
        """
        SELECT
            paper_id,
            paper_uid,
            canonical_source,
            source_order
        FROM papers
        WHERE source_order NOT LIKE (
            '["' || canonical_source || '"%'
        )
        LIMIT 100
        """,
        conn,
    )

source_order_issues

,paper_id,paper_uid,canonical_source,source_order


In [36]:
DROP_STAGING_TABLE = True

In [37]:
if DROP_STAGING_TABLE:
    with sqlite3.connect(CENTRAL_DB_PATH) as conn:
        conn.execute("DROP TABLE IF EXISTS staging_records")
        conn.execute("VACUUM")
        conn.commit()

    print("Staging table removed and database compacted.")
else:
    print("Staging table retained for inspection.")

Staging table removed and database compacted.
